# SweetTV - TV Program Recommender (Item-based CF kNN, implicit feedback = screen_time)

Notebook này:
1) Implement Item-based Collaborative Filtering kNN (item-item) với implicit feedback là **screen_time**.
2) Đánh giá trên `logs_val.parquet` với Precision/Recall/F1/NDCG/MAP @K (default K=5).
3) Train final (train+val) và tạo `submission_itemcf_knn.csv` theo format Kaggle.
4) Khi tạo submission: chỉ recommend trong **candidate set** từ `metadata_test.parquet`, thiếu thì fallback bằng popular items (cũng thuộc candidate).

Input paths (theo yêu cầu):
- /content/logs_train.parquet
- /content/logs_val.parquet
- /content/metadata_test.parquet
- /content/submission.csv

Output:
- /content/submission_itemcf_knn.csv


In [ ]:
# ============ 0. Setup ============
!pip -q install pyarrow tqdm scipy

import os
import gc
import time
import math
import random
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Dict, Tuple, List, Set, Optional

from tqdm import tqdm
from scipy import sparse

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)

class Timer:
    def __init__(self, name: str):
        self.name = name
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        dt = time.perf_counter() - self.t0
        print(f"[TIMER] {self.name}: {dt:.2f}s")

def assert_no_nan_inf_sparse(mat: sparse.spmatrix, name: str):
    if mat.nnz == 0:
        return
    d = mat.data
    assert np.isfinite(d).all(), f"{name} has NaN/Inf!"

SEED = 42
set_seed(SEED)
print("Seed =", SEED)


Seed = 42


## 0.1 Config
Bạn có thể thay đổi:
- K (topK recommend, mặc định 5)
- K_neighbors (số láng giềng item-item, mặc định 200)
- REL_THRESH (ngưỡng relevance trên val, mặc định 0.8)
- AGG_METHOD (sum hoặc mean cho implicit rating)


In [ ]:
# ============ Config ============

TRAIN_PATH = "/content/drive/MyDrive/Project/TV_Recommender/preprocessed_data2/logs_train.parquet"
VAL_PATH   = "/content/drive/MyDrive/Project/TV_Recommender/preprocessed_data2/logs_val.parquet"
SUBMISSION_PATH   = "/content/drive/MyDrive/Project/TV_Recommender/data/submission.csv"
META_TEST_PATH = "/content/drive/MyDrive/Project/TV_Recommender/preprocessed_data2/metadata_test.parquet"

K_RECO = 5
K_NEIGHBORS = 100

REL_THRESH = 0.5        # default 0.8, can change to 0.5 etc.
AGG_METHOD = "sum"      # "sum" or "mean"

EPS = 1e-9

print("K_RECO =", K_RECO)
print("K_NEIGHBORS =", K_NEIGHBORS)
print("REL_THRESH =", REL_THRESH)
print("AGG_METHOD =", AGG_METHOD)


K_RECO = 5
K_NEIGHBORS = 100
REL_THRESH = 0.5
AGG_METHOD = sum


# 1. Load data (train/val + submission + metadata_test) & EDA

Yêu cầu in ra:
- số dòng train/val
- #users unique, #items unique (tv_show_id != 0)
- % tv_show_id == 0 (train/val)
- #candidate items trong metadata_test (tv_show_id != 0)


In [ ]:
# ============ 1. Load data ============
def load_data(
    train_path: str,
    val_path: str,
    meta_test_path: str,
    submission_path: str
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    for p in [train_path, val_path, meta_test_path, submission_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing file: {p}")

    with Timer("Read parquet/csv"):
        train_df = pd.read_parquet(train_path)
        val_df   = pd.read_parquet(val_path)
        meta_test = pd.read_parquet(meta_test_path)
        sub_df = pd.read_csv(submission_path, dtype={"user_id": "uint64", "tv_show_id": "string"})

    return train_df, val_df, meta_test, sub_df


train_df, val_df, meta_test, sub_df = load_data(TRAIN_PATH, VAL_PATH, META_TEST_PATH, SUBMISSION_PATH)

def basic_eda(train_df: pd.DataFrame, val_df: pd.DataFrame, meta_test: pd.DataFrame):
    def _eda_one(df: pd.DataFrame, name: str):
        n = len(df)
        pct0 = (df["tv_show_id"] == 0).mean() * 100.0
        df_nz = df[df["tv_show_id"] != 0]
        n_users = df_nz["user_id"].nunique()
        n_items = df_nz["tv_show_id"].nunique()
        print(f"--- {name} ---")
        print("Rows:", n)
        print("Unique users (tv_show_id!=0):", n_users)
        print("Unique items (tv_show_id!=0):", n_items)
        print("% tv_show_id == 0:", f"{pct0:.2f}%")
        print()

    _eda_one(train_df, "TRAIN")
    _eda_one(val_df, "VAL")

    cand = meta_test.loc[meta_test["tv_show_id"] != 0, "tv_show_id"].unique()
    print("--- METADATA_TEST ---")
    print("Candidate items (tv_show_id!=0):", len(cand))

basic_eda(train_df, val_df, meta_test)


[TIMER] Read parquet/csv: 1.86s
--- TRAIN ---
Rows: 1878969
Unique users (tv_show_id!=0): 4838
Unique items (tv_show_id!=0): 3716
% tv_show_id == 0: 50.41%

--- VAL ---
Rows: 525983
Unique users (tv_show_id!=0): 4657
Unique items (tv_show_id!=0): 1897
% tv_show_id == 0: 50.22%

--- METADATA_TEST ---
Candidate items (tv_show_id!=0): 6636


# 2. Preprocess + Build implicit matrix (sparse)

- Lọc tv_show_id != 0
- Aggregate implicit rating theo (user_id, tv_show_id):
  r_ui = sum(screen_time) (default) hoặc mean(screen_time)
- Tạo mapping user2idx, item2idx dựa trên TRAIN (phase eval) / TRAIN+VAL (phase final)
- Build sparse matrix R (CSR) shape [n_users, n_items], values float32
- Ground truth cho evaluation:
  - aggregate val theo (user_id, tv_show_id): r_ui_val = sum(screen_time)
  - gt_u = {item | r_ui_val >= REL_THRESH}
  - cold-item handling: chỉ giữ gt items có trong train item2idx
  - report cold-item rate


In [ ]:
# ============ 2. Preprocess ============
def preprocess_logs(
    df: pd.DataFrame,
    agg_method: str = "sum"
) -> pd.DataFrame:
    """
    Filter tv_show_id != 0 and aggregate screen_time by (user_id, tv_show_id).
    Returns DataFrame columns: [user_id, tv_show_id, r_ui]
    """
    df = df[df["tv_show_id"] != 0].copy()

    if agg_method not in ["sum", "mean"]:
        raise ValueError("agg_method must be 'sum' or 'mean'")

    if agg_method == "sum":
        agg = df.groupby(["user_id", "tv_show_id"], as_index=False)["screen_time"].sum()
    else:
        agg = df.groupby(["user_id", "tv_show_id"], as_index=False)["screen_time"].mean()

    agg = agg.rename(columns={"screen_time": "r_ui"})
    agg["r_ui"] = agg["r_ui"].astype(np.float32)
    return agg

def build_mappings(
    interactions: pd.DataFrame
) -> Tuple[Dict[int,int], Dict[int,int], np.ndarray, np.ndarray]:
    """
    Build user2idx, item2idx from interactions DataFrame with columns [user_id, tv_show_id, r_ui]
    """
    users = interactions["user_id"].unique()
    items = interactions["tv_show_id"].unique()

    users = np.array(users, dtype=np.uint64)
    items = np.array(items, dtype=np.int64)

    user2idx = {int(u): i for i, u in enumerate(users)}
    item2idx = {int(it): j for j, it in enumerate(items)}

    return user2idx, item2idx, users, items

def build_sparse_matrix(
    interactions: pd.DataFrame,
    user2idx: Dict[int,int],
    item2idx: Dict[int,int],
    n_users: int,
    n_items: int
) -> sparse.csr_matrix:
    """
    Build CSR matrix R from interactions.
    """
    u = interactions["user_id"].map(lambda x: user2idx[int(x)]).to_numpy(dtype=np.int32)
    i = interactions["tv_show_id"].map(lambda x: item2idx[int(x)]).to_numpy(dtype=np.int32)
    v = interactions["r_ui"].to_numpy(dtype=np.float32)

    R = sparse.csr_matrix((v, (u, i)), shape=(n_users, n_items), dtype=np.float32)
    R.sum_duplicates()
    R.eliminate_zeros()
    return R

# Preprocess train/val interactions
with Timer("Preprocess train interactions"):
    train_agg = preprocess_logs(train_df, agg_method=AGG_METHOD)

with Timer("Preprocess val interactions (sum for GT)"):
    val_agg_sum = preprocess_logs(val_df, agg_method="sum")

# Build mappings from TRAIN only for evaluation
with Timer("Build mappings (TRAIN)"):
    user2idx_tr, item2idx_tr, idx2user_tr, idx2item_tr = build_mappings(train_agg)

n_users_tr = len(idx2user_tr)
n_items_tr = len(idx2item_tr)
print("n_users_tr =", n_users_tr, "| n_items_tr =", n_items_tr)

with Timer("Build sparse matrix R_train"):
    R_train = build_sparse_matrix(train_agg, user2idx_tr, item2idx_tr, n_users_tr, n_items_tr)

print("R_train shape:", R_train.shape, "nnz:", R_train.nnz, "density:", R_train.nnz/(R_train.shape[0]*R_train.shape[1]))
assert_no_nan_inf_sparse(R_train, "R_train")


[TIMER] Preprocess train interactions: 0.37s
[TIMER] Preprocess val interactions (sum for GT): 0.17s
[TIMER] Build mappings (TRAIN): 0.02s
n_users_tr = 4838 | n_items_tr = 3716
[TIMER] Build sparse matrix R_train: 0.59s
R_train shape: (4838, 3716) nnz: 397840 density: 0.02212925925942407


# 3. Train Item-based CF kNN (item-item)

Cosine similarity giữa vector item trên không gian user:

- Normalize item columns:
  norm_i = sqrt(sum_u r_ui^2) + eps
- X = R * diag(1/norm_i)
- Co-occurrence/cosine:
  S = X.T @ X
- Set diag=0
- Giữ topK_neighbors cho mỗi item để tiết kiệm RAM (CSR)

Không loop i,j toàn bộ I^2.


In [ ]:
# ============ 3. Item-item similarity ============
def _topk_rows_csr(mat: sparse.csr_matrix, k: int) -> sparse.csr_matrix:
    """
    Keep top-k largest values per row in CSR matrix.
    This loops over rows (items) but not over full I^2.
    """
    mat = mat.tocsr()
    indptr = mat.indptr
    indices = mat.indices
    data = mat.data

    new_indptr = np.zeros(mat.shape[0] + 1, dtype=np.int64)
    new_indices_list = []
    new_data_list = []

    for r in range(mat.shape[0]):
        start, end = indptr[r], indptr[r+1]
        row_nnz = end - start
        if row_nnz == 0:
            new_indptr[r+1] = new_indptr[r]
            continue

        row_data = data[start:end]
        row_idx = indices[start:end]

        if row_nnz > k:
            # partial select top-k by value
            topk_pos = np.argpartition(row_data, -k)[-k:]
            # sort descending
            topk_sorted = topk_pos[np.argsort(-row_data[topk_pos])]
            row_data = row_data[topk_sorted]
            row_idx  = row_idx[topk_sorted]
        else:
            # sort descending
            order = np.argsort(-row_data)
            row_data = row_data[order]
            row_idx = row_idx[order]

        new_indices_list.append(row_idx.astype(np.int32))
        new_data_list.append(row_data.astype(np.float32))
        new_indptr[r+1] = new_indptr[r] + len(row_data)

    new_indices = np.concatenate(new_indices_list) if len(new_indices_list) else np.array([], dtype=np.int32)
    new_data = np.concatenate(new_data_list) if len(new_data_list) else np.array([], dtype=np.float32)

    out = sparse.csr_matrix((new_data, new_indices, new_indptr), shape=mat.shape, dtype=np.float32)
    out.eliminate_zeros()
    return out

def compute_item_similarity_topk(
    R: sparse.csr_matrix,
    k_neighbors: int = 200,
    eps: float = 1e-9
) -> sparse.csr_matrix:
    """
    Compute cosine similarity S (item-item) and keep topK per item.
    Returns CSR matrix S shape [n_items, n_items] (row i: neighbors of item i).
    """
    n_users, n_items = R.shape

    # item norms
    with Timer("Compute item norms"):
        # sum of squares per column
        col_sq = np.array(R.power(2).sum(axis=0)).ravel().astype(np.float64)
        norms = np.sqrt(col_sq) + eps
        inv_norms = (1.0 / norms).astype(np.float32)

    with Timer("Normalize columns (R @ D_inv)"):
        D_inv = sparse.diags(inv_norms, offsets=0, format="csr", dtype=np.float32)
        X = (R @ D_inv).tocsr()  # normalized columns

    with Timer("Compute full cosine sim (X.T @ X)"):
        S = (X.T @ X).tocsr().astype(np.float32)

    # zero diagonal
    with Timer("Zero diagonal"):
        S.setdiag(0.0)
        S.eliminate_zeros()

    assert_no_nan_inf_sparse(S, "S_full")

    with Timer(f"Keep top-{k_neighbors} per item"):
        S_topk = _topk_rows_csr(S, k=k_neighbors)

    assert_no_nan_inf_sparse(S_topk, "S_topk")
    return S_topk

with Timer("Train similarity on TRAIN"):
    S_train = compute_item_similarity_topk(R_train, k_neighbors=K_NEIGHBORS, eps=EPS)

print("S_train shape:", S_train.shape, "nnz:", S_train.nnz, "avg nnz/row:", S_train.nnz / S_train.shape[0])


[TIMER] Compute item norms: 0.00s
[TIMER] Normalize columns (R @ D_inv): 0.01s
[TIMER] Compute full cosine sim (X.T @ X): 0.58s
[TIMER] Zero diagonal: 0.01s
[TIMER] Keep top-100 per item: 0.10s
[TIMER] Train similarity on TRAIN: 0.71s
S_train shape: (3716, 3716) nnz: 369858 avg nnz/row: 99.53121636167923


# 4. Scoring & Recommend

Với user u:
- profile r_u (sparse row)
- scores = r_u @ S
- seen-filter: loại item user đã xem trong TRAIN
- cold-user fallback: recommend top popular items theo train (sum r_ui)

Các hàm bắt buộc:
- recommend_user_topk()


In [ ]:
# ============ 4. Recommend ============
def compute_popular_items(
    train_interactions: pd.DataFrame,
    item2idx: Dict[int,int],
    candidate_tv_set: Optional[Set[int]] = None
) -> List[int]:
    """
    Return list of item indices (mapped) sorted by popularity (sum r_ui), descending.
    If candidate_tv_set provided, only keep items in candidate set.
    """
    df = train_interactions
    if candidate_tv_set is not None:
        df = df[df["tv_show_id"].isin(candidate_tv_set)]

    pop = df.groupby("tv_show_id", as_index=False)["r_ui"].sum()
    pop = pop.sort_values("r_ui", ascending=False)

    pop_item_idx = []
    for it in pop["tv_show_id"].to_list():
        it = int(it)
        if it in item2idx:
            pop_item_idx.append(item2idx[it])
    return pop_item_idx

def recommend_user_topk(
    user_id: int,
    R: sparse.csr_matrix,
    S: sparse.csr_matrix,
    user2idx: Dict[int,int],
    idx2item: np.ndarray,
    popular_item_idx: List[int],
    k_reco: int = 5,
    candidate_item_mask: Optional[np.ndarray] = None,   # boolean mask over items (mapped space)
    seen_filter: bool = True
) -> List[int]:
    """
    Recommend top-k tv_show_id for a user_id.
    - candidate_item_mask: only allow items where mask=True (used for submission)
    """
    n_items = S.shape[0]

    if user_id not in user2idx:
        # cold user
        return _fallback_popular([], idx2item, popular_item_idx, k_reco, candidate_item_mask)

    uidx = user2idx[user_id]
    r_u = R[uidx]  # 1 x n_items CSR
    seen_idx = set(r_u.indices.tolist()) if seen_filter else set()

    if r_u.nnz == 0:
        return _fallback_popular(seen_idx, idx2item, popular_item_idx, k_reco, candidate_item_mask)

    # score: 1 x n_items (sparse)
    scores_sparse = r_u @ S
    scores = np.asarray(scores_sparse.todense()).ravel().astype(np.float32)  # n_items

    # seen-filter
    if seen_filter and len(seen_idx) > 0:
        scores[list(seen_idx)] = -np.inf

    # candidate filter
    if candidate_item_mask is not None:
        scores[~candidate_item_mask] = -np.inf

    # pick top-k
    # handle case all -inf
    finite_mask = np.isfinite(scores)
    if not finite_mask.any():
        return _fallback_popular(seen_idx, idx2item, popular_item_idx, k_reco, candidate_item_mask)

    # argpartition top-k among finite
    k_eff = min(k_reco, int(finite_mask.sum()))
    top_idx = np.argpartition(scores, -k_eff)[-k_eff:]
    top_idx = top_idx[np.argsort(-scores[top_idx])]

    rec_item_ids = []
    used = set()
    for j in top_idx:
        if not np.isfinite(scores[j]):
            continue
        if j in used:
            continue
        used.add(int(j))
        rec_item_ids.append(int(idx2item[j]))
        if len(rec_item_ids) == k_reco:
            break

    # fill if needed
    if len(rec_item_ids) < k_reco:
        rec_item_ids = _fill_with_popular(rec_item_ids, used, seen_idx, idx2item, popular_item_idx, k_reco, candidate_item_mask)

    return rec_item_ids

def _fallback_popular(seen_idx: Set[int], idx2item: np.ndarray, popular_item_idx: List[int], k_reco: int,
                      candidate_item_mask: Optional[np.ndarray]) -> List[int]:
    rec = []
    used = set()
    for j in popular_item_idx:
        if j in seen_idx:
            continue
        if candidate_item_mask is not None and not candidate_item_mask[j]:
            continue
        if j in used:
            continue
        used.add(j)
        rec.append(int(idx2item[j]))
        if len(rec) == k_reco:
            return rec
    return rec

def _fill_with_popular(current: List[int], used_item_idx: Set[int], seen_idx: Set[int],
                       idx2item: np.ndarray, popular_item_idx: List[int], k_reco: int,
                       candidate_item_mask: Optional[np.ndarray]) -> List[int]:
    rec = current[:]
    for j in popular_item_idx:
        if j in seen_idx:
            continue
        if j in used_item_idx:
            continue
        if candidate_item_mask is not None and not candidate_item_mask[j]:
            continue
        used_item_idx.add(j)
        rec.append(int(idx2item[j]))
        if len(rec) == k_reco:
            break
    return rec

# Popular items (for eval stage): from TRAIN agg, no candidate restriction
with Timer("Compute popular items (TRAIN)"):
    popular_idx_tr = compute_popular_items(train_agg, item2idx_tr, candidate_tv_set=None)

print("Popular items prepared:", len(popular_idx_tr))


[TIMER] Compute popular items (TRAIN): 0.01s
Popular items prepared: 3716


# 5. Evaluate on val (Precision/Recall/F1/NDCG/MAP @K)

- GT từ val_agg_sum: gt_u = {item | r_ui_val >= REL_THRESH}
- Cold-item handling: chỉ giữ gt items có trong train item2idx
- Chỉ tính metrics cho user có gt_u không rỗng sau lọc
- Báo cáo:
  - num_users_eval
  - tỷ lệ user bị bỏ qua (gt rỗng)
  - cold-item gt rate
  - bảng metrics @K


In [ ]:
# ============ 5. Evaluate ============
def build_ground_truth(
    val_agg_sum: pd.DataFrame,
    item2idx_train: Dict[int,int],
    rel_thresh: float = 0.8
) -> Tuple[Dict[int, Set[int]], float]:
    """
    Build ground truth as dict: user_id -> set(item_idx_in_train_space)
    Drops val items not in train item2idx; returns cold-item drop rate among GT positives.
    """
    # relevant interactions
    rel = val_agg_sum[val_agg_sum["r_ui"] >= rel_thresh].copy()
    if len(rel) == 0:
        return {}, 0.0

    # map items to train space if possible
    rel["item_idx"] = rel["tv_show_id"].map(lambda x: item2idx_train.get(int(x), -1)).astype(np.int32)
    total_rel = len(rel)
    kept = rel[rel["item_idx"] >= 0]
    dropped = total_rel - len(kept)
    cold_item_rate = dropped / max(total_rel, 1)

    gt = {}
    for uid, grp in kept.groupby("user_id"):
        gt[int(uid)] = set(grp["item_idx"].tolist())
    return gt, cold_item_rate

def _metrics_one_user(pred_item_idx: List[int], gt_item_idx: Set[int], k: int) -> Tuple[float,float,float,float,float]:
    """
    pred_item_idx: list of item_idx (train space) length k (or less)
    gt_item_idx: set of item_idx
    Returns: P, R, F1, NDCG, AP
    """
    if len(gt_item_idx) == 0:
        return 0,0,0,0,0

    pred = pred_item_idx[:k]
    hit = [1 if i in gt_item_idx else 0 for i in pred]

    # Precision, Recall
    P = sum(hit) / k
    R = sum(hit) / len(gt_item_idx)
    F1 = 0.0 if (P+R)==0 else (2*P*R)/(P+R)

    # NDCG
    dcg = 0.0
    for t, rel in enumerate(hit, start=1):
        if rel:
            dcg += 1.0 / math.log2(t + 1)
    idcg = 0.0
    for t in range(1, min(k, len(gt_item_idx)) + 1):
        idcg += 1.0 / math.log2(t + 1)
    ndcg = 0.0 if idcg == 0 else dcg / idcg

    # AP@K
    ap_sum = 0.0
    hit_cnt = 0
    for t, rel in enumerate(hit, start=1):
        if rel:
            hit_cnt += 1
            ap_sum += hit_cnt / t
    denom = min(k, len(gt_item_idx))
    ap = 0.0 if denom == 0 else ap_sum / denom

    return P, R, F1, ndcg, ap

def evaluate_metrics_at_k(
    R_train: sparse.csr_matrix,
    S_train: sparse.csr_matrix,
    user2idx_train: Dict[int,int],
    idx2item_train: np.ndarray,
    train_agg: pd.DataFrame,
    val_agg_sum: pd.DataFrame,
    item2idx_train: Dict[int,int],
    k: int = 5,
    rel_thresh: float = 0.8,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Evaluate metrics on val.
    """
    gt, cold_item_rate = build_ground_truth(val_agg_sum, item2idx_train, rel_thresh=rel_thresh)

    # Popular items in train space for fallback
    popular_idx = compute_popular_items(train_agg, item2idx_train, candidate_tv_set=None)

    users_all = sorted(gt.keys())
    num_gt_users = len(users_all)

    if num_gt_users == 0:
        print("No GT users found with given REL_THRESH.")
        return pd.DataFrame()

    P_list, R_list, F1_list, NDCG_list, AP_list = [], [], [], [], []
    skipped = 0

    with Timer(f"Evaluation loop @K={k}"):
        for uid in tqdm(users_all, total=len(users_all)):
            gt_u = gt.get(uid, set())
            if len(gt_u) == 0:
                skipped += 1
                continue

            # recommend -> tv_show_id list, then map to item_idx for metric
            rec_item_ids = recommend_user_topk(
                user_id=uid,
                R=R_train,
                S=S_train,
                user2idx=user2idx_train,
                idx2item=idx2item_train,
                popular_item_idx=popular_idx,
                k_reco=k,
                candidate_item_mask=None,
                seen_filter=True
            )
            # map back to item_idx in train space
            rec_item_idx = [item2idx_train.get(int(it), -1) for it in rec_item_ids]
            rec_item_idx = [x for x in rec_item_idx if x >= 0]

            P, Rm, F1, ndcg, ap = _metrics_one_user(rec_item_idx, gt_u, k)
            P_list.append(P)
            R_list.append(Rm)
            F1_list.append(F1)
            NDCG_list.append(ndcg)
            AP_list.append(ap)

    num_eval = len(P_list)
    skip_rate = 1 - (num_eval / max(num_gt_users, 1))

    metrics = {
        f"Precision@{k}": float(np.mean(P_list)) if num_eval else 0.0,
        f"Recall@{k}": float(np.mean(R_list)) if num_eval else 0.0,
        f"F1@{k}": float(np.mean(F1_list)) if num_eval else 0.0,
        f"NDCG@{k}": float(np.mean(NDCG_list)) if num_eval else 0.0,
        f"MAP@{k}": float(np.mean(AP_list)) if num_eval else 0.0,
        "num_users_eval": num_eval,
        "skip_rate_gt_empty": float(skip_rate),
        "cold_item_gt_rate": float(cold_item_rate),
    }

    out = pd.DataFrame([metrics])
    if verbose:
        print("\n=== EVAL SUMMARY ===")
        print(out.T)
    return out

eval_df_k5 = evaluate_metrics_at_k(
    R_train=R_train,
    S_train=S_train,
    user2idx_train=user2idx_tr,
    idx2item_train=idx2item_tr,
    train_agg=train_agg,
    val_agg_sum=val_agg_sum,
    item2idx_train=item2idx_tr,
    k=K_RECO,
    rel_thresh=REL_THRESH,
    verbose=True
)

# Optional: evaluate K=10
eval_df_k10 = evaluate_metrics_at_k(
    R_train=R_train,
    S_train=S_train,
    user2idx_train=user2idx_tr,
    idx2item_train=idx2item_tr,
    train_agg=train_agg,
    val_agg_sum=val_agg_sum,
    item2idx_train=item2idx_tr,
    k=10,
    rel_thresh=REL_THRESH,
    verbose=False
)
print("\nOptional @10:")
display(eval_df_k10.T)


100%|██████████| 4575/4575 [00:04<00:00, 1032.39it/s]


[TIMER] Evaluation loop @K=5: 4.44s

=== EVAL SUMMARY ===
                              0
Precision@5            0.113705
Recall@5               0.034420
F1@5                   0.045107
NDCG@5                 0.122039
MAP@5                  0.072271
num_users_eval      4575.000000
skip_rate_gt_empty     0.000000
cold_item_gt_rate      0.177313


100%|██████████| 4575/4575 [00:07<00:00, 644.76it/s]

[TIMER] Evaluation loop @K=10: 7.11s

Optional @10:


,0
Precision@10,0.088634
Recall@10,0.053509
F1@10,0.055544
NDCG@10,0.107087
MAP@10,0.049427
num_users_eval,4575.000000
skip_rate_gt_empty,0.000000
cold_item_gt_rate,0.177313


# 6. Train FINAL (train+val) + tạo submission.csv

Bắt buộc:
- Train similarity trên logs_train + logs_val (lọc tv_show_id!=0)
- Candidate set:
  candidate_tv_ids = metadata_test[tv_show_id != 0].unique().tolist()
  candidate_tv_set = set(candidate_tv_ids)
- Với mỗi user_id trong submission.csv:
  - recommend top 5 theo model + seen-filter + candidate_filter
  - nếu chưa đủ 5 -> fill bằng popular unseen (cũng thuộc candidate_tv_set)
- Output đúng format:
  user_id (uint64), tv_show_id (object, "id1 id2 id3 id4 id5")
- Sanity check:
  - mỗi dòng đúng 5 id
  - không có 0
  - không trùng item trong 1 dòng


In [ ]:
# ============ 6. Train final + Submission ============

def make_candidate_set(meta_test: pd.DataFrame) -> Tuple[List[int], Set[int]]:
    cand_ids = meta_test.loc[meta_test["tv_show_id"] != 0, "tv_show_id"].dropna().astype(np.int64).unique().tolist()
    cand_set = set(map(int, cand_ids))
    return cand_ids, cand_set

def make_submission(
    sub_df: pd.DataFrame,
    R: sparse.csr_matrix,
    S: sparse.csr_matrix,
    user2idx: Dict[int,int],
    idx2item: np.ndarray,
    item2idx: Dict[int,int],
    popular_item_idx: List[int],
    candidate_tv_set: Set[int],
    k: int = 5
) -> pd.DataFrame:
    """
    Create submission dataframe with [user_id(uint64), tv_show_id(object)].
    Recommend only in candidate_tv_set (intersection with known items).
    """
    # Build candidate mask in mapped item space
    n_items = len(idx2item)
    candidate_mask = np.zeros(n_items, dtype=bool)
    cand_in_model = 0
    for it in candidate_tv_set:
        j = item2idx.get(int(it), None)
        if j is not None:
            candidate_mask[j] = True
            cand_in_model += 1
    print(f"Candidate items in metadata_test: {len(candidate_tv_set)} | in model space: {cand_in_model}")

    user_ids = sub_df["user_id"].astype(np.uint64).to_numpy()
    out_rows = []

    with Timer("Generate recommendations for submission"):
        for uid in tqdm(user_ids, total=len(user_ids)):
            uid_int = int(uid)
            rec = recommend_user_topk(
                user_id=uid_int,
                R=R,
                S=S,
                user2idx=user2idx,
                idx2item=idx2item,
                popular_item_idx=popular_item_idx,
                k_reco=k,
                candidate_item_mask=candidate_mask,
                seen_filter=True
            )

            # If still not enough (rare), fill using candidate items (even if unseen) - as last resort
            if len(rec) < k:
                # fallback using candidate ids that are in model and not already in rec
                rec_set = set(rec)
                for j in popular_item_idx:
                    it = int(idx2item[j])
                    if it in candidate_tv_set and it not in rec_set:
                        rec.append(it)
                        rec_set.add(it)
                    if len(rec) == k:
                        break

            # ensure exactly k
            rec = rec[:k]
            out_rows.append(" ".join(map(str, rec)))

    out = pd.DataFrame({
        "user_id": user_ids.astype(np.uint64),
        "tv_show_id": pd.Series(out_rows, dtype="object")
    })
    return out

def sanity_check_submission(df: pd.DataFrame, k: int = 5):
    assert df["user_id"].dtype == np.uint64, f"user_id dtype must be uint64, got {df['user_id'].dtype}"
    assert df["tv_show_id"].dtype == object, f"tv_show_id dtype must be object, got {df['tv_show_id'].dtype}"

    bad = 0
    for s in df["tv_show_id"].tolist():
        parts = str(s).split()
        if len(parts) != k:
            bad += 1
            continue
        ids = [int(x) for x in parts]
        if any(x == 0 for x in ids):
            bad += 1
            continue
        if len(set(ids)) != k:
            bad += 1
            continue
    print("Sanity check bad rows:", bad, "/", len(df))
    assert bad == 0, "Submission has invalid rows!"

# ---- Train FINAL on train+val ----
with Timer("Preprocess train+val interactions"):
    trainval_df = pd.concat([train_df, val_df], ignore_index=True)
    trainval_agg = preprocess_logs(trainval_df, agg_method=AGG_METHOD)

with Timer("Build mappings (TRAIN+VAL)"):
    user2idx_f, item2idx_f, idx2user_f, idx2item_f = build_mappings(trainval_agg)

n_users_f, n_items_f = len(idx2user_f), len(idx2item_f)
print("n_users_f =", n_users_f, "| n_items_f =", n_items_f)

with Timer("Build sparse matrix R_trainval"):
    R_trainval = build_sparse_matrix(trainval_agg, user2idx_f, item2idx_f, n_users_f, n_items_f)

with Timer("Train similarity on TRAIN+VAL"):
    S_final = compute_item_similarity_topk(R_trainval, k_neighbors=K_NEIGHBORS, eps=EPS)

# candidate set from metadata_test
candidate_tv_ids, candidate_tv_set = make_candidate_set(meta_test)
print("candidate_tv_set size:", len(candidate_tv_set))

# popular items restricted to candidate set for submission fallback
with Timer("Compute popular items (TRAIN+VAL, restricted to candidate)"):
    popular_idx_final = compute_popular_items(trainval_agg, item2idx_f, candidate_tv_set=candidate_tv_set)

print("popular_idx_final size:", len(popular_idx_final))

# make submission
sub_out = make_submission(
    sub_df=sub_df,
    R=R_trainval,
    S=S_final,
    user2idx=user2idx_f,
    idx2item=idx2item_f,
    item2idx=item2idx_f,
    popular_item_idx=popular_idx_final,
    candidate_tv_set=candidate_tv_set,
    k=K_RECO
)

print(sub_out.head())
print(sub_out.dtypes)

# save
OUT_PATH = "/content/submission_itemcf_knn.csv"
sub_out.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

# sanity checks
sanity_check_submission(sub_out, k=K_RECO)


[TIMER] Preprocess train+val interactions: 0.71s
[TIMER] Build mappings (TRAIN+VAL): 0.02s
n_users_f = 4876 | n_items_f = 4159
[TIMER] Build sparse matrix R_trainval: 1.48s
[TIMER] Compute item norms: 0.00s
[TIMER] Normalize columns (R @ D_inv): 0.03s
[TIMER] Compute full cosine sim (X.T @ X): 1.26s
[TIMER] Zero diagonal: 0.03s
[TIMER] Keep top-100 per item: 0.26s
[TIMER] Train similarity on TRAIN+VAL: 1.60s
candidate_tv_set size: 6636
[TIMER] Compute popular items (TRAIN+VAL, restricted to candidate): 0.05s
popular_idx_final size: 3022
Candidate items in metadata_test: 6636 | in model space: 3022


100%|██████████| 1260/1260 [00:00<00:00, 1355.68it/s]

[TIMER] Generate recommendations for submission: 0.94s
               user_id                              tv_show_id
0  8377619604347126107    5900347 200432 700382 700389 2500413
1  8381667675275833309    20088 10002514 400426 200477 2400508
2  8387147770138767246  6700482 400335 700389 5900347 90079063
3  8397181578236218580    200344 200477 5900348 6000483 400335
4  8404698046253197367   6000483 400335 5900347 5900348 400426
user_id       uint64
tv_show_id    object
dtype: object
Saved: /content/submission_itemcf_knn.csv
Sanity check bad rows: 0 / 1260


# 7. Save outputs + download + extra sanity

- Download file để nộp Kaggle
- Assert: similarity không NaN/Inf
- Gợi ý: nếu bạn chạy nhiều lần, có thể giảm `K_NEIGHBORS` để nhanh hơn / ít RAM hơn


In [ ]:
# ============ 7. Download ============
from google.colab import files

# extra sanity: similarity
assert_no_nan_inf_sparse(S_final, "S_final")
print("S_final OK (finite).")

files.download("/content/submission_itemcf_knn.csv")


S_final OK (finite).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>